# 01 · CRECls

Clusters the putative distal CREs (pdCREs) of each species by their accessibility across cell states. The CRECls are
used by the figures of the later notebooks.

1. **Peaks**: mouse and rabbit embryonic peaks and their locus ids.
2. **pdCREs**: drop peaks with N in their sequence, TSS peaks and CTCF peaks.
3. **Accessibility**: ATAC UMIs of each pdCRE in each mouse SMC and rabbit metacell of an embryonic cell type, as
   log2(UMI fraction + 10⁻⁵), the fraction taken over all peaks.
4. **CRECls**: k-means (tglkmeans, k = 100, seed 19) over the pdCREs of each species.

## Setup

In [1]:
source("../R/setup.R")
source("../R/cell_types.R")
out_dir = out_path("01_crecls")

## 1. Peaks

Each embryonic peak (`cre_id`) is a 500-bp locus of the species namespace; joining on the interval gives its locus id.

In [2]:
mm = fread(data_path("mm_emb_locif.csv")) %>%
    left_join(fread(data_path("mm_namespace_2025.csv")), by = c("chrom", "start", "end"))
oc = fread(data_path("oc_emb2_locif.csv")) %>%
    left_join(fread(data_path("oc_namespace_2025.csv")), by = c("chrom", "start", "end"))
orth_full = fread(data_path("orth_2025.csv"))

## 2. pdCREs

Motif energies are missing for loci whose sequence contains N (assembly gaps); these peaks are dropped. Of the rest, TSS
peaks and then CTCF peaks are set aside (Methods):

- **TSS**: mouse, an annotated TSS within 500 bp or strong H3K4me3; rabbit, an annotated start codon within 500 bp, the
  rabbit image of a mouse TSS within 500 bp, or a sequence ortholog of a mouse TSS peak.
- **CTCF**: a CTCF motif score above −20. Mouse also: within 500 bp of an ENCODE CTCF peak, or an ES-cell CTCF ChIP-seq
  maximum above 32 (`mm_ctcf_evidence.csv`). Rabbit also: a sequence
  ortholog of a mouse peak within 500 bp of an ENCODE CTCF peak.

The remaining peaks are the pdCREs.

In [3]:
complete_energy_loci = function(file, id) {
    energies = fread(data_path(file))
    motifs = energies %>% select(-chrom, -start, -end, -all_of(id))
    energies[[id]][rowSums(is.na(motifs)) == 0]
}
mm = mm %>% filter(mm_locid %in% complete_energy_loci("mm_energies_raw_2025.csv", "mm_locid"))
oc = oc %>% filter(oc_locid %in% complete_energy_loci("oc_energies_raw_2025.csv", "oc_locid"))

In [4]:
mm_tss = fread(data_path("mm_namespace_tss_only.csv"))
mm_ctcf = fread(data_path("mm_namespace_ctcf_only.csv"))
more_mm_ctcf = fread(data_path("mm_ctcf_evidence.csv")) %>% filter(abs(dist) < 500 | ctcf_chip_max > 32)
more_mm_ctcf_orth = more_mm_ctcf %>%
    left_join(orth_full %>% select(mm_peak_id, mm_locid, oc_locid, oc_peak_id), by = "mm_locid") %>%
    filter(abs(dist) < 500, !is.na(oc_peak_id))
oc_tss = fread(data_path("oc_namespace_tss_only.csv"))
oc_tss_imgs = fread(data_path("oc_namespace_tss_imgs_only.csv"))
oc_tss_orth = fread(data_path("oc_tss_orth.csv"))
oc_ctcf = fread(data_path("oc_namespace_ctcf_only.csv"))

mm_class = case_when(
    mm$mm_locid %in% mm_tss$mm_locid ~ "TSS",
    mm$mm_locid %in% c(mm_ctcf$mm_locid, more_mm_ctcf$mm_locid) ~ "CTCF",
    TRUE ~ "pdCRE"
)
oc_class = case_when(
    oc$oc_locid %in% c(oc_tss$oc_locid, oc_tss_imgs$oc_locid, oc_tss_orth$oc_locid) ~ "TSS",
    oc$oc_locid %in% c(oc_ctcf$oc_locid, more_mm_ctcf_orth$oc_locid) ~ "CTCF",
    TRUE ~ "pdCRE"
)
mm_notss = mm %>% filter(mm_class == "pdCRE")
oc_notss = oc %>% filter(oc_class == "pdCRE")

## 3. Accessibility

Mouse metacells are summed into SMCs (metacell groups, Methods). A mouse SMC or rabbit metacell is used if its type is
embryonic (`R/cell_types.R`) and it has more than 350,000 (mouse) or 250,000 (rabbit) ATAC UMIs in peaks. Accessibility
is log2(UMI fraction + 10⁻⁵), the fraction of the unit's UMIs over all peaks (TSS and CTCF peaks included); the k-means
uses the pdCRE rows.

In [5]:
mc_type_mm = fread(data_path("mc_type_mm_2025.csv"))
mc_type_oc = fread(data_path("metacell_types-2024-12-04.csv")) %>%
    mutate(metacell = gsub(".[0-9]+$", "", gsub("^M", "mc", metacell))) %>%
    filter(color %in% oc_type_colors) %>%
    arrange(match(color, oc_type_colors))
smc_type_mm = with(distinct(mc_type_mm, smc, smc_type), setNames(smc_type, smc))
mm_idx2smc = setNames(mc_type_mm$smc, mc_type_mm$index)
oc_idx2sh = setNames(mc_type_oc$cell_type, mc_type_oc$metacell)
mm_col_order = mc_type_mm %>%
    filter(color %in% mm_type_colors) %>%
    arrange(match(color, mm_type_colors)) %>%
    pull(cell_type) %>%
    unique()
oc_col_order = unique(mc_type_oc$cell_type)

In [6]:
mm_atac_m = as.matrix(fread(data_path("mc_atac_mat_2025.csv")) %>% column_to_rownames("mm_locid") %>% select(-chrom, -start, -end))
mm_smc_atacf = t(col_group_sums(mm_atac_m[, names(mm_idx2smc)], mm_idx2smc))[mm$mm_locid, ]
oc_atac_m = as.matrix(fread(data_path("gotg_mc_p_multi_full.csv")) %>% column_to_rownames("oc_locid"))
oc_mc_atacf = oc_atac_m[oc$oc_locid, ]

mm_f3 = smc_type_mm[colnames(mm_smc_atacf)] %in% mm_col_order & colSums(mm_smc_atacf) > 350000
oc_f3 = oc_idx2sh[colnames(oc_mc_atacf)] %in% oc_col_order & colSums(oc_mc_atacf) > 250000
mm_a_legc = log2(1e-5 + t(t(mm_smc_atacf[, mm_f3]) / colSums(mm_smc_atacf[, mm_f3])))[mm_notss$mm_locid, ]
oc_a_legc = log2(1e-5 + t(t(oc_mc_atacf[, oc_f3]) / colSums(oc_mc_atacf[, oc_f3])))[oc_notss$oc_locid, ]

units = bind_rows(
    tibble(species = "mouse", unit = colnames(mm_smc_atacf), type = unname(smc_type_mm[unit]),
           embryonic = type %in% mm_col_order, umis_in_peaks = unname(colSums(mm_smc_atacf)), used = unname(mm_f3)),
    tibble(species = "rabbit", unit = colnames(oc_mc_atacf), type = unname(oc_idx2sh[unit]),
           embryonic = type %in% oc_col_order, umis_in_peaks = unname(colSums(oc_mc_atacf)), used = unname(oc_f3))
)

## 4. CRECls

`TGL_kmeans` (tglkmeans 0.5.5) calls `set.seed(seed)` and runs k-means++ on R's random number generator; the CRECl ids are
then ordered by hierarchical clustering of the cluster centers. With this version, the CRECls are fixed by the seed and
the matrix above, including its row order.

In [7]:
mm_km = TGL_kmeans(mm_a_legc, k = 100, id_column = FALSE, seed = 19)
oc_km = TGL_kmeans(oc_a_legc, k = 100, id_column = FALSE, seed = 19)

## Outputs

For the later notebooks: the CRECl (`kmc`) of each pdCRE (`mm_crecl.csv`, `oc_crecl.csv`), the mouse pdCREs
(`mm_notss_noctcf.csv`), the units with their type, ATAC UMIs in peaks and use (`units.csv`), the clustered matrices
(`accessibility.rds`) and the ATAC UMIs of every peak in the used units (`umis.rds`).

In [8]:
fwrite(tibble::enframe(mm_km$cluster) %>% rename(locid = name, kmc = value), file.path(out_dir, "mm_crecl.csv"))
fwrite(tibble::enframe(oc_km$cluster) %>% rename(locid = name, kmc = value), file.path(out_dir, "oc_crecl.csv"))
fwrite(mm_notss, file.path(out_dir, "mm_notss_noctcf.csv"))
fwrite(units, file.path(out_dir, "units.csv"))
saveRDS(list(mm_a_legc = mm_a_legc, oc_a_legc = oc_a_legc), file.path(out_dir, "accessibility.rds"))
saveRDS(list(mm_smc_atacf = mm_smc_atacf[, mm_f3], oc_mc_atacf = oc_mc_atacf[, oc_f3]), file.path(out_dir, "umis.rds"), compress = FALSE)